# C8-neural-training — Losses, Gradients, and Training (Unit Overview)

This is the index notebook for the five 90-minute teaching sessions in
`lessons/`. C6 supplied network architecture and forward passes; C7 supplied
PyTorch tensors, `nn.Module`, parameters, and inspection. C8 now joins those
ideas into a complete, auditable training system.

## Unit goals

By the end of the unit, you can:

- turn a batch of logits of shape `(N, C)` into stable softmax probabilities,
  evaluate categorical cross-entropy, derive the softmax Jacobian, and derive
  the fused logit gradient `(p - y) / N`;
- backpropagate through a two-layer MLP by hand, with every intermediate shape
  and every accumulated gradient stated;
- implement and certify a seeded NumPy MLP training loop from forward cache
  through backward pass and parameter update;
- train the same model with PyTorch autograd and an optimizer, using the exact
  lifecycle `zero_grad → forward → loss → backward → step`;
- distinguish parameters from buffers and training behavior from deterministic
  evaluation behavior for BatchNorm and dropout.

## Prerequisites recap

The declared prerequisites are **F3-multivar-calculus**,
**C3-gradient-descent**, **C6-neural-networks**, and **C7-pytorch**.

From F3, recall partial derivatives, gradients, and the multivariable chain
rule. From C3, recall learning rate, stochastic gradient descent, loss
surfaces, and the update `parameter -= learning_rate * gradient`. From C6,
recall ReLU, MLP architecture, matrix shapes, initialization variance,
overfitting, and L2 regularization. From C7, recall torch tensors,
`nn.Module`, `nn.Parameter`, `requires_grad`, parameter counting, and
Python inheritance. F1/F5 vocabulary used below includes seeded NumPy arrays,
broadcasting, aggregation axes, expectation, and variance.

## Five-session map

| Session | Notebook | Main contract |
|---|---|---|
| 1 | `01-softmax-and-cross-entropy.ipynb` | stable logits → probabilities → loss, Jacobian, fused gradient |
| 2 | `02-manual-backpropagation.ipynb` | dependency order, shape ledger, accumulation, complete two-layer backward pass |
| 3 | `03-numpy-mlp-training.ipynb` | cache → backward → update, gradient check, trained-vs-untrained behavior |
| 4 | `04-pytorch-autograd-and-optimizers.ipynb` | autograd lifecycle, accumulation, optimizer state, deterministic evaluation |
| 5 | `05-batchnorm-and-dropout.ipynb` | batch/running statistics, affine state, inverted dropout, train/eval audit |

Every assertion that compares floating-point values names `atol` and `rtol`.
Every random example uses seed **20260804**. All code is CPU-small and uses no
download, pretrained model, external API, or hidden data.

## How to study

1. Before running a code cell, predict its shapes and at least one numerical
   invariant.
2. Answer each checkpoint on paper. Answers are collected only at the end of
   that session.
3. Keep a four-column audit beside every training loop: **mode**, **gradient
   state**, **optimizer action**, **evaluation context**.
4. After Session 5, use `review.ipynb` without reopening the lessons; then
   return to the lesson named in the weak-spot table.

## Forward link

C10 reuses this complete training lifecycle for convolutional networks.
Convolution changes the feature extractor and its shape arithmetic; it does
not change the loss, backpropagation logic, optimizer ordering, BatchNorm/
dropout mode contract, or evaluation discipline taught here.


In [ ]:
import numpy as np
import torch

SEED = 20260804
np.random.default_rng(SEED)
torch.manual_seed(SEED)  # torch.manual_seed: seed torch's global RNG
# torch.set_default_dtype: new float tensors default to float64
torch.set_default_dtype(torch.float64)

print("NumPy:", np.__version__)
print("PyTorch:", torch.__version__)
# torch.get_default_dtype(): the dtype that new float tensors get
print("seed:", SEED, "| default torch dtype:", torch.get_default_dtype())

<!-- api-card -->
## Code in this unit

A quick reference for the library calls in this unit's code. Each one is explained in a comment (or in the text beside the code) where it first appears.

**New in this unit:**

- `torch.get_default_dtype` — the dtype new float tensors get
- `np.log` — the natural logarithm of each entry
- `np.outer` — the matrix with `(i, j)` entry `p[i] * p[j]`
- `torch.use_deterministic_algorithms` — with `True`, every operation must be reproducible (or raise an error)
- `.square` — each entry squared
- `.backward` — runs autograd from a scalar loss, filling each parameter's `.grad`
- `torch.allclose` — True when every pair of entries is close (`atol`, `rtol`)
- `.detach` — the same values, cut off from the autograd graph
- `nn.Linear` — an affine layer `x @ W.T + b` from `in` to `out` features
- `torch.relu` — `max(0, x)` for each entry
- `.manual_seed` — `generator.manual_seed(s)`: seeds a separate random generator
- `torch.Generator` — a private random generator, independent of the global one
- `nn.CrossEntropyLoss` — mean softmax cross-entropy of logits against integer labels
- `torch.optim.SGD` — gradient-descent optimizer (optional `momentum`) over the given parameters
- `.repeat` — tiles a tensor, e.g. `.repeat((24, 1))` stacks 24 copies of the rows
- `.zero_grad` — clears the old gradients before the next backward pass
- `.step` — one optimizer update of every parameter, using the current gradients
- `torch.linalg.vector_norm` — the Euclidean length of a tensor
- `torch.optim.Adam` — Adam optimizer: adaptive per-parameter steps with base rate `lr`
- `torch.optim.lr_scheduler.OneCycleLR` — learning-rate schedule up to `max_lr` and back down over `total_steps`
- `.train` — puts a module in training mode
- `.eval` — puts a module in evaluation mode (changes dropout and batch norm)
- `torch.no_grad` — a `with` block in which autograd records nothing
- `torch.isfinite` — True where an entry is not `inf` or `nan`
- `torch.sqrt` — the square root of each entry
- `torch.rsqrt` — `1 / sqrt(x)` for each entry
- `.requires_grad_` — turns gradient tracking on for this tensor, in place
- `nn.BatchNorm1d` — batch normalization of each feature over the batch
- `.named_buffers` — `(name, tensor)` pairs of a module's non-learned state, e.g. running means
- `nn.Dropout` — in training mode zeros each entry with probability `p` and rescales the rest
- `torch.isclose` — True where two tensors' entries are close
- `torch.linspace` — evenly spaced points from `a` to `b`, both ends included
- `torch.abs` — the absolute value of each entry
- `np.full_like` — an array shaped like the given one, every entry one value
- `.update` — adds (or overwrites) dict entries
- `.standard_normal` — `rng.standard_normal(shape)`: draws from `N(0, 1)`
- `nn.ReLU` — the `max(0, x)` activation as a layer
- `torch.empty` — an unfilled tensor of the given shape, to be written into
- `.cpu` — a copy of the tensor on the CPU
- `torch.cat` — joins tensors along an existing axis
- `.repeat_interleave` — repeats each entry in place: `[0, 1]` becomes `[0, 0, 1, 1]`
- `.double` — the tensor converted to float64

**Also used here (met earlier):**

- `np.linalg.norm` — Euclidean length (first explained in F2)
- `.state_dict` — module tensors keyed by name (first explained in C7)